# Tic-Tac-Toe Structural Estimation — Full Demo
This notebook shows data generation, estimation with **Newton–Armijo** and **NRLS**, visualizations, and human-vs-bot play.
Dependency: `numpy` only.

In [ ]:
import numpy as np
from pathlib import Path
from tictactoe_play_utils import (FEATURE_NAMES, features, logits_for_state,
    play_game, print_trajectory, human_vs_bot, board_to_str, X, O)
from run_estimation import newton_maximize, generate_dataset
from nrls_optimizer import nrls_maximize, loglik
np.set_printoptions(precision=3, suppress=True)

## Generate dataset

In [ ]:
theta_star = np.array([2.0,3.0,0.5,0.3,0.0,0.6])
dataset, theta_star = generate_dataset(n_games=1500, theta_star=theta_star, seed=42)
len(dataset), theta_star

## Newton–Armijo MLE

In [9]:
theta0 = np.zeros(len(FEATURE_NAMES))
theta_hat_newton, ll_newton = newton_maximize(dataset, theta0, lam=1e-6, max_iter=8, tol=1e-6)
theta_hat_newton, ll_newton

iter 01: ll=-18674.650 ||g||=3.452e+03 theta=[0. 0. 0. 0. 0. 0.]
iter 02: ll=-14152.312 ||g||=2.430e+02 theta=[ 1.181  3.036  0.248 -0.029 -0.219  0.417]
iter 03: ll=-14100.651 ||g||=2.553e+01 theta=[ 2.094  3.041  0.228  0.014 -0.242  0.593]
iter 04: ll=-14100.123 ||g||=3.539e-01 theta=[ 2.048  3.068  0.229  0.015 -0.244  0.599]
iter 05: ll=-14100.123 ||g||=7.681e-05 theta=[ 2.048  3.068  0.229  0.015 -0.244  0.599]
iter 06: ll=-14100.123 ||g||=6.721e-05 theta=[ 2.048  3.068  0.229  0.015 -0.244  0.599]
iter 07: ll=-14100.123 ||g||=5.041e-05 theta=[ 2.048  3.068  0.229  0.015 -0.244  0.599]
iter 08: ll=-14100.123 ||g||=5.036e-05 theta=[ 2.048  3.068  0.229  0.015 -0.244  0.599]


(array([ 2.048,  3.068,  0.229,  0.015, -0.244,  0.599]),
 np.float64(-14100.122933017421))

## NRLS (Nested Recursive Lexicographical Search)

In [4]:
bounds = [(-4,6),(-4,6),(-2,2),(-2,2),(-2,2),(-2,2)]
f = lambda th: loglik(dataset, np.asarray(th), lam=1e-6)
res = nrls_maximize(f, bounds, levels=(5,7,9), topk=6, shrink=0.4)
res.theta, res.value, res.evaluations


== Level 1/3 | grid=5, topk=6, shrink=0.4 ==
  - Dimension 0 / 5
  - Dimension 1 / 5
  - Dimension 2 / 5
  - Dimension 3 / 5
  - Dimension 4 / 5
  - Dimension 5 / 5
  >> level best so far: val=-14298.6418, theta=[3.5 3.5 0.  0.  0.  1. ]

== Level 2/3 | grid=7, topk=6, shrink=0.4 ==
  - Dimension 0 / 5
  - Dimension 1 / 5
  - Dimension 2 / 5
  - Dimension 3 / 5
  - Dimension 4 / 5
  - Dimension 5 / 5
  >> level best so far: val=-14127.2637, theta=[ 2.167  2.833  0.267  0.    -0.267  0.733]

== Level 3/3 | grid=9, topk=6, shrink=0.4 ==
  - Dimension 0 / 5
  - Dimension 1 / 5
  - Dimension 2 / 5
  - Dimension 3 / 5
  - Dimension 4 / 5
  - Dimension 5 / 5
  >> level best so far: val=-14101.0470, theta=[ 1.967  3.033  0.187  0.    -0.267  0.573]


(array([ 1.967,  3.033,  0.187,  0.   , -0.267,  0.573]),
 np.float64(-14101.04699932078),
 665)

## Visualize a trajectory as 3×3 boards

In [10]:
traj, w = play_game(theta_star, first_player=X, seed=0, stochastic=False)
print_trajectory(traj)
print('Winner:', {0:'draw',1:'X',2:'O'}[w])

Move 1: Player X -> 4
. . .
. . .
. . .

Move 2: Player O -> 0
. . .
. X .
. . .

Move 3: Player X -> 2
O . .
. X .
. . .

Move 4: Player O -> 6
O . X
. X .
. . .

Move 5: Player X -> 3
O . X
. X .
O . .

Move 6: Player O -> 5
O . X
X X .
O . .

Move 7: Player X -> 1
O . X
X X O
O . .

Move 8: Player O -> 7
O X X
X X O
O . .

Move 9: Player X -> 8
O X X
X X O
O O .

Winner: draw


## Play the bot (interactive)

In [11]:
human_vs_bot(theta_hat_newton, bot_player='O', stochastic=False)  # uncomment to play

Index map:
0 1 2
3 4 5
6 7 8

Start:
. . .
. . .
. . .

Bot plays 4:
. . .
. O X
. . .

Bot plays 8:
. . X
. O X
. . O

Bot plays 1:
X O X
. O X
. . O

Bot plays 6:
X O X
. O X
O X O

Draw!

Final board:
X O X
X O X
O X O


0

## Save dataset to CSV

In [ ]:
# import csv
# def board_to_string(board): return ''.join('.XO'[v] for v in board)
# csv_path = Path('tictactoe_dataset.csv')
# with open(csv_path, 'w', newline='') as f:
#     w = csv.writer(f); w.writerow(['board','player','action'])
#     for b,p,a in dataset: w.writerow([board_to_string(b), p, a])
# csv_path.resolve()